# Host taxonomies

In [1]:
import pandas as pd
from Bio import SeqIO

samples = [
    'AP1','AP2','AP3','AP4','AP5',
    'BP1','BP2','BP3','BP4','BP5',
    'MP1','MP2','MP3','MP4','MP5',
    'PP1','PP2','PP3','PP4','PP5',
]

def change_name(contig_id, samplename):
    return f'{samplename}_{contig_id}'

def parse_samples(samplelist):
    df = pd.DataFrame()
    temp_df = pd.DataFrame()
    for sample in samplelist:
        temp_df = pd.read_csv(f"/MP_Data/Dados_David/viral_analysis/results/{sample}/iphop/Host_prediction_to_genome_m90.csv",sep=',')
        temp_df['Virus'] = temp_df['Virus'].apply(lambda x: change_name(x, sample))
        if df.empty:
            df = temp_df
        else:
            df = pd.concat([df,temp_df])
    return df

iphop_df = parse_samples(samples)

# CHecking if the votus with predicted hosts are inside the final filtered votus
viral_list = pd.read_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/tables/rel_abund/votus_dataset1_TMM_rel_abund.tsv", sep='\t', header=None)[0].tolist()

#Removing NCLDVs and virophages (IPHOP does not not contain eukaryotic host data)
ncldv_list = pd.read_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/ncldvs_virophages/ncldv_scaffolds.tsv", sep='\t', header = None)[0].tolist()
virophage_list = pd.read_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/ncldvs_virophages/virophage_clust_df.tsv", sep='\t', header = None)[0].tolist()

for votu in viral_list:
    if votu in ncldv_list or votu in virophage_list:
        viral_list.remove(votu)

iphop_df = iphop_df[iphop_df['Virus'].isin(viral_list)]

iphop_df = (
    iphop_df.sort_values("Confidence score", ascending=False)
      .drop_duplicates(subset=["Virus"], keep="first")
)
iphop_df.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/05_host_prediction/iphop_results.tsv',sep='\t',index=False)
iphop_df

,Virus,Host genome,Host taxonomy,Main method,Confidence score,Additional methods
849,BP4_NODE_177_length_45190_cov_6.909272,RS_GCF_910585845.1,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,CRISPR,99.0,NaN
1034,MP2_NODE_357_length_40675_cov_3.096750,RS_GCF_023824315.1,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,CRISPR,99.0,iPHoP-RF;73.70
1240,BP4_NODE_2753_length_13067_cov_7.695666,RS_GCF_900200035.1,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,CRISPR,98.9,blast;90.10 iPHoP-RF;62.20
1123,BP4_NODE_2585_length_13572_cov_81.117778,RS_GCF_001984825.2,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,CRISPR,98.9,iPHoP-RF;81.40
1174,MP1_NODE_522_length_44735_cov_4.291607,GB_GCA_900539195.1,d__Bacteria;p__Bacillota;c__Clostridia;o__Osci...,CRISPR,98.9,NaN
...,...,...,...,...,...,...
630,PP3_NODE_966_length_16818_cov_2.403032,GB_GCA_947447375.1,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,iPHoP-RF,90.1,NaN
1140,MP5_NODE_3740_length_10484_cov_2.016607,RS_GCF_019797845.1,d__Bacteria;p__Pseudomonadota;c__Gammaproteoba...,iPHoP-RF,90.1,NaN
152,PP4_NODE_2471_length_10561_cov_2.085681,GB_GCA_025455635.1,d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__...,iPHoP-RF,90.1,NaN
863,BP4_NODE_1843_length_16290_cov_10.100031,GB_GCA_016887505.1,d__Bacteria;p__Bacillota;c__TC1;o__TC1;f__UBA5...,blast,90.0,NaN


# Investigating the most abundant host families

In [2]:
iphop_df = iphop_df[['Virus','Host taxonomy']]
iphop_df[['Domain',"Phylum", "Class", "Order", "Family", "Genus", "Species"]] = iphop_df['Host taxonomy'].str.split(';', n=7, expand = True)
iphop_df = iphop_df.drop("Host taxonomy", axis=1)
iphop_df['Domain'].value_counts()

/MP_Data/database/rgi/wildcard/custom_tmp/ipykernel_2832003/3673015708.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iphop_df[['Domain',"Phylum", "Class", "Order", "Family", "Genus", "Species"]] = iphop_df['Host taxonomy'].str.split(';', n=7, expand = True)
/MP_Data/database/rgi/wildcard/custom_tmp/ipykernel_2832003/3673015708.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iphop_df[['Domain',"Phylum", "Class", "Order", "Family", "Genus", "Species"]] = iphop_df['Host taxonomy'].str.split(';', n=7,

Domain
d__Bacteria    703
d__Archaea      28
Name: count, dtype: int64

In [3]:
print(f'Phylumns: {iphop_df['Phylum'].unique()}, {len(iphop_df['Phylum'].unique())}')
print(f'Classes: {iphop_df['Class'].unique()}, {len(iphop_df['Class'].unique())}')
print(f'Orders: {iphop_df['Order'].unique()}, {len(iphop_df['Order'].unique())}')
print(f'Families: {iphop_df['Family'].unique()}, {len(iphop_df['Family'].unique())}')
print(f'Genus: {iphop_df['Genus'].unique()}, {len(iphop_df['Genus'].unique())}')

Phylumns: ['p__Bacteroidota' 'p__Pseudomonadota' 'p__Bacillota'
 'p__Patescibacteriota' 'p__Thermoplasmatota' 'p__Myxococcota'
 'p__Actinomycetota' 'p__Nitrospirota' 'p__Planctomycetota'
 'p__Verrucomicrobiota' 'p__Acidobacteriota' 'p__Cyanobacteriota'
 'p__Chloroflexota' 'p__Halobacteriota' 'p__UBA3054' 'p__Spirochaetota'
 'p__Asgardarchaeota' 'p__Nanobdellota' 'p__Chlamydiota'
 'p__Caldisericota' 'p__Thermoproteota' 'p__Armatimonadota'
 'p__Omnitrophota' 'p__Aenigmatarchaeota' 'p__Bacteroidota_A'], 25
Classes: ['c__Bacteroidia' 'c__Gammaproteobacteria' 'c__Clostridia' 'c__Bacilli'
 'c__Negativicutes' 'c__Saccharimonadia' 'c__Poseidoniia' 'c__Polyangia'
 'c__Actinomycetes' 'c__Nitrospiria' 'c__Planctomycetia'
 'c__Alphaproteobacteria' 'c__Verrucomicrobiia' 'c__Minisyncoccia'
 'c__Terriglobia' 'c__Cyanobacteriia' 'c__Anaerolineae'
 'c__Acidimicrobiia' 'c__Methanosarcinia' 'c__UBA3054' 'c__Spirochaetia'
 'c__Phycisphaerae' 'c__E2' 'c__Heimdallarchaeia' 'c__Nanobdellia'
 'c__Thermoplasma

# Checking how many votus with classified hosts and unclassified in taxonomy

In [4]:
tax_df = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/tables/taxonomy/final_taxonomy.tsv',sep='\t')
tax_df
vir_tax = pd.merge(iphop_df, tax_df, left_on='Virus', right_on='seq_name',how='left')
vir_tax['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                             655
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                  61
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae              8
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae                 2
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Straboviridae                  2
Viruses;Monodnaviria;Sangervirae;Phixviricota;Malgrandaviricetes;Petitvirales;Microviridae      1
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Casjensviridae                 1
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;                   1
Name: count, dtype: int64

# Raw number of contigs assigned to each baterial phylum and family

In [5]:
def get_sample_name(contig_id):
    return contig_id.split("_")[0]

def define_city(sample):
    if sample.startswith("AP"):
        return "Altamira"
    elif sample.startswith("BP"):
        return "Bragança"
    elif sample.startswith("MP"):
        return "Maraba"
    elif sample.startswith("PP"):
        return "Paragominas"

#Phylum

phylum = iphop_df[['Virus', "Phylum"]]
phylum['sample'] = phylum['Virus'].apply(get_sample_name)
phylum = phylum.drop('Virus',axis=1)
phylum = phylum.groupby('sample').value_counts()
phylum = phylum.to_frame()
phylum = phylum.reset_index()
phylum = pd.pivot_table(phylum, index='Phylum', columns='sample', values='count').fillna(0)
phylum['total'] = phylum.sum(axis=1)
phylum = phylum.sort_values(by='total', ascending=False)
phylum = phylum.drop('total', axis=1)
phylum

/MP_Data/database/rgi/wildcard/custom_tmp/ipykernel_2832003/3922281139.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  phylum['sample'] = phylum['Virus'].apply(get_sample_name)


sample,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,BP5,MP1,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5
Phylum,,,,,,,,,,,,,,,,,,,,
p__Pseudomonadota,8.0,10.0,27.0,12.0,27.0,4.0,16.0,13.0,68.0,8.0,75.0,16.0,24.0,15.0,24.0,1.0,11.0,7.0,9.0,9.0
p__Bacteroidota,4.0,10.0,3.0,1.0,11.0,0.0,1.0,0.0,1.0,0.0,6.0,10.0,4.0,5.0,13.0,0.0,2.0,2.0,3.0,1.0
p__Cyanobacteriota,7.0,7.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,1.0,1.0,5.0,6.0,11.0,17.0,2.0,0.0,0.0,0.0,0.0
p__Bacillota,1.0,3.0,2.0,1.0,4.0,1.0,0.0,0.0,2.0,0.0,3.0,5.0,3.0,5.0,9.0,1.0,0.0,0.0,0.0,1.0
p__Planctomycetota,5.0,7.0,0.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0,4.0,6.0,2.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0
p__Actinomycetota,1.0,1.0,2.0,0.0,3.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0,1.0,5.0,5.0,1.0,0.0,1.0,1.0,0.0
p__Verrucomicrobiota,1.0,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,2.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0
p__Patescibacteriota,0.0,2.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,0.0,4.0,0.0,1.0,0.0,0.0,0.0
p__Thermoplasmatota,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,4.0,3.0,0.0,0.0,0.0,0.0,2.0


In [6]:
phylum.sum(axis=1) / phylum.sum(axis=1).sum() * 100

Phylum
p__Pseudomonadota       52.530780
p__Bacteroidota         10.533516
p__Cyanobacteriota       9.575923
p__Bacillota             5.608755
p__Planctomycetota       4.377565
p__Actinomycetota        3.419973
p__Verrucomicrobiota     2.599179
p__Patescibacteriota     2.051984
p__Thermoplasmatota      1.915185
p__Acidobacteriota       1.504788
p__Chloroflexota         1.367989
p__Myxococcota           1.094391
p__Nanobdellota          0.683995
p__Nitrospirota          0.547196
p__Thermoproteota        0.410397
p__Aenigmatarchaeota     0.273598
p__Asgardarchaeota       0.273598
p__Halobacteriota        0.273598
p__Caldisericota         0.136799
p__Armatimonadota        0.136799
p__Bacteroidota_A        0.136799
p__Omnitrophota          0.136799
p__Chlamydiota           0.136799
p__Spirochaetota         0.136799
p__UBA3054               0.136799
dtype: float64

In [7]:
#Family
family = iphop_df[['Virus', "Family"]]
family['sample'] = family['Virus'].apply(get_sample_name)
family = family.drop('Virus',axis=1)
family = family.groupby('sample').value_counts()
family = family.to_frame()
family = family.reset_index()
family = pd.pivot_table(family, index='Family', columns='sample', values='count').fillna(0)
family['total'] = family.sum(axis=1)
family = family.sort_values(by='total', ascending=False)
family = family.head(15)
family = family.drop('total', axis=1)
family

/MP_Data/database/rgi/wildcard/custom_tmp/ipykernel_2832003/2706586420.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  family['sample'] = family['Virus'].apply(get_sample_name)


sample,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,BP5,MP1,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5
Family,,,,,,,,,,,,,,,,,,,,
f__Burkholderiaceae,0.0,2.0,13.0,6.0,14.0,0.0,7.0,11.0,9.0,3.0,16.0,7.0,12.0,6.0,8.0,1.0,4.0,4.0,4.0,3.0
f__Cyanobiaceae,7.0,5.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,5.0,8.0,13.0,1.0,0.0,0.0,0.0,0.0
f__Enterobacteriaceae,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,43.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
f__Flavobacteriaceae,2.0,3.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,1.0,4.0,0.0,1.0,1.0,0.0,0.0
f__Sphingomonadaceae,2.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,4.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.0,1.0
f__Bacteroidaceae,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,3.0,0.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0
f__Moraxellaceae,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,7.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
f__Beijerinckiaceae,2.0,2.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
f__Xanthobacteraceae,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
family.sum(axis=1) / family.sum(axis=1).sum() * 100

Family
f__Burkholderiaceae      33.942559
f__Cyanobiaceae          14.099217
f__Enterobacteriaceae    12.271540
f__Flavobacteriaceae      5.483029
f__Sphingomonadaceae      4.438642
f__Bacteroidaceae         3.655352
f__Moraxellaceae          3.394256
f__Beijerinckiaceae       3.394256
f__Xanthobacteraceae      3.133159
f__Rhodocyclaceae         3.133159
f__UBA1268                3.133159
f__Chitinophagaceae       2.872063
f__Gallionellaceae        2.610966
f__Pseudomonadaceae       2.349869
f__Lachnospiraceae        2.088773
dtype: float64

# Add mapping data

In [9]:
iphop_family = iphop_df[['Virus','Family']]
viral_abund = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/tables/rel_abund/votus_dataset1_TMM_rel_abund.tsv', sep='\t')

viral_abund_family = pd.merge(iphop_family, viral_abund, left_on='Virus', right_on='contig_id', how='left')
viral_abund_family = viral_abund_family.drop('contig_id',axis=1)
viral_abund_family

,Virus,Family,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
0,BP4_NODE_177_length_45190_cov_6.909272,f__Bacteroidaceae,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.317313,0.108798,...,0.152660,0.105908,0.175176,0.000000,0.000000,0.106693,0.048382,0.107333,0.162054,0.356575
1,MP2_NODE_357_length_40675_cov_3.096750,f__Moraxellaceae,0.000000,0.000000,0.051719,0.000000,0.000000,0.000000,0.055861,0.000000,...,0.000000,0.000000,0.031411,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,BP4_NODE_2753_length_13067_cov_7.695666,f__Enterobacteriaceae,0.000000,0.000000,0.000000,0.000000,0.000000,0.055327,0.055861,0.000000,...,0.000000,0.000000,0.000000,0.057595,0.031174,0.291009,0.138644,0.477381,0.373200,0.065457
3,BP4_NODE_2585_length_13572_cov_81.117778,f__Enterobacteriaceae,0.000000,0.000000,0.000000,0.000000,0.000000,0.055327,0.317313,0.252581,...,0.000000,0.201667,0.091419,0.112053,0.250744,0.156053,0.402606,0.621177,0.487535,0.516688
4,MP1_NODE_522_length_44735_cov_4.291607,f__Ruminococcaceae,0.000000,0.000000,0.076609,0.000000,0.000000,0.000000,0.274583,0.296255,...,0.000000,0.000000,0.000000,0.000000,0.061405,0.027759,0.000000,0.022408,0.000000,0.044110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,PP3_NODE_966_length_16818_cov_2.403032,f__Flavobacteriaceae,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.711188,1.630360,1.903862,1.742785,1.493226,1.331484,1.513890,0.923579,0.779875,0.744266
727,MP5_NODE_3740_length_10484_cov_2.016607,f__Enterobacteriaceae,0.000000,0.050736,0.051719,0.000000,0.150422,0.000000,0.000000,0.055878,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
728,PP4_NODE_2471_length_10561_cov_2.085681,f__Saprospiraceae,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.606588,1.489616,1.401994,1.315645,1.493226,1.009846,1.367297,0.967589,0.968342,0.461386
729,BP4_NODE_1843_length_16290_cov_10.100031,f__UBA5767,0.000000,0.099021,0.026194,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.057595,0.409877,0.269744,0.402606,0.819346,0.929422,0.865200


In [10]:
final_tax_by_sample = pd.DataFrame({
    'Taxonomy': viral_abund_family['Family'],

    # AP group
    'AP1': viral_abund_family.iloc[:, 2:5].mean(axis=1),   
    'AP2': viral_abund_family.iloc[:, 5:8].mean(axis=1),   
    'AP3': viral_abund_family.iloc[:, 8:11].mean(axis=1),  
    'AP4': viral_abund_family.iloc[:, 11:14].mean(axis=1), 
    'AP5': viral_abund_family.iloc[:, 14:17].mean(axis=1), 

    # BP group
    'BP1': viral_abund_family.iloc[:, 17:20].mean(axis=1),                  
    'BP2': viral_abund_family.iloc[:, 20:23].mean(axis=1), 
    'BP3': viral_abund_family.iloc[:, 23:26].mean(axis=1),
    'BP4': viral_abund_family.iloc[:, 26:29].mean(axis=1),
    'BP5': viral_abund_family.iloc[:, 29:32].mean(axis=1),

    # MP group
    'MP1': viral_abund_family.iloc[:, 32:35].mean(axis=1),
    'MP2': viral_abund_family.iloc[:, 35:38].mean(axis=1),
    'MP3': viral_abund_family.iloc[:, 38:41].mean(axis=1),
    'MP4': viral_abund_family.iloc[:, 41:44].mean(axis=1),
    'MP5': viral_abund_family.iloc[:, 44:47].mean(axis=1),

    # PP group
    'PP1': viral_abund_family.iloc[:, 47:50].mean(axis=1),
    'PP2': viral_abund_family.iloc[:, 50:53].mean(axis=1),
    'PP3': viral_abund_family.iloc[:, 53:56].mean(axis=1),
    'PP4': viral_abund_family.iloc[:, 56:59].mean(axis=1),
    'PP5': viral_abund_family.iloc[:, 59:62].mean(axis=1)
})
final_tax_by_sample = final_tax_by_sample.groupby('Taxonomy').sum()
final_tax_by_sample

,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,BP5,MP1,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5
Taxonomy,,,,,,,,,,,,,,,,,,,,
f__2-12-FULL-35-15,0.000000,0.000000,0.052692,0.059694,0.015635,0.000000,0.020646,0.037927,0.000000,0.000000,2.152988,0.000000,0.000000,0.000000,0.000000,0.018336,0.000000,0.000000,0.000000,0.014775
f__2-12-FULL-49-11,0.341546,0.352190,0.311239,0.276577,0.043430,0.526004,0.673125,0.447483,0.231037,0.197037,2.419268,0.085265,0.091109,0.106617,0.058310,0.191326,0.046887,0.144563,0.164555,0.115577
f__ARS21,0.008731,0.101517,0.000000,0.000000,0.242826,0.000000,0.000000,0.000000,0.000000,0.000000,0.885869,1.612714,1.253682,1.508418,2.018649,0.018606,0.054161,0.000000,0.000000,0.000000
f__Acetobacteraceae,5.662307,7.105238,1.190743,1.356295,1.764145,4.458106,2.539536,1.730720,1.440578,1.025836,4.590985,7.601946,7.258945,7.192883,8.033790,0.344547,0.415306,0.492404,0.727518,0.678749
f__Acidaminococcaceae,0.000000,0.021127,3.440395,3.499978,1.768899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.025408,0.007232,0.451885,0.007957,0.044405,0.018655,0.000000,0.000000,0.014775
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f__Verrucomicrobiaceae,0.000000,0.000000,0.164916,0.137086,0.247586,0.276502,0.164262,0.177376,0.037878,0.032417,0.835287,2.353764,2.055675,2.144288,2.639561,0.019550,0.000000,0.020622,0.016127,0.008274
f__Vibrionaceae,0.099382,0.066393,2.153526,1.549902,3.258359,0.298987,0.607625,0.554179,3.429830,0.581406,2.558795,5.648407,5.181390,6.431561,8.088186,0.009918,0.045445,0.109444,0.470852,0.734773
f__Vicingaceae,1.511552,2.103819,0.708789,0.625875,0.453387,0.000000,0.000000,0.011189,0.000000,0.000000,0.054798,0.077446,0.098503,0.144123,0.154356,0.038016,0.000000,0.030359,0.057272,0.008274


In [11]:
final_tax_by_sample['total'] = final_tax_by_sample.sum(axis=1)
final_tax_by_sample_top15 = final_tax_by_sample.sort_values(by='total',ascending=False)
final_tax_by_sample_top15 = final_tax_by_sample_top15.drop('total',axis=1)
#final_tax_by_sample_top15 = final_tax_by_sample_top15.head(15)
final_tax_by_sample_top15

,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,BP5,MP1,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5
Taxonomy,,,,,,,,,,,,,,,,,,,,
f__Burkholderiaceae,15.719413,17.610139,112.588145,96.049953,112.997331,52.883655,108.270427,126.314790,140.987595,110.268837,88.904423,85.411906,101.302642,92.242552,105.632335,44.419093,62.475478,60.905680,70.970494,73.072034
f__Cyanobiaceae,50.452415,49.071818,9.160954,8.346671,80.373993,0.951660,1.464714,1.013306,0.824753,0.927009,59.558148,87.500577,84.195431,106.901541,119.651564,6.867891,4.900831,4.563884,3.020628,1.690448
f__Enterobacteriaceae,0.308530,0.431493,5.384053,5.284144,3.927752,5.220354,11.645119,17.360977,158.375984,13.869092,9.402922,1.709022,2.403795,3.084424,4.007356,2.813909,1.381374,2.452144,8.502463,17.367226
f__Sphingomonadaceae,11.271593,12.276063,12.586772,11.886782,11.810503,12.844183,12.769449,14.658605,15.748798,25.183064,17.924756,10.698383,11.882519,12.295578,12.703599,7.279409,6.527913,6.777799,12.371086,9.872205
f__Rhodocyclaceae,0.646211,0.751449,4.029321,3.679333,6.963040,5.674488,14.292007,11.225736,11.535435,8.025880,12.524950,10.022697,14.438589,11.003996,12.170511,1.652195,0.983401,0.942363,3.539209,5.681492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
f__Haliangiaceae,0.000000,0.018442,0.115086,0.067423,0.056715,0.076365,0.137531,0.140756,0.000000,0.024604,1.781559,0.012745,0.013957,0.090336,0.119639,0.044405,0.017839,0.029804,0.000000,0.037752
f__Rhodanobacteraceae,0.000000,0.009349,0.054235,0.018173,0.009277,0.149054,0.135715,0.144340,0.077982,0.024219,1.782723,0.033584,0.007232,0.000000,0.080914,0.009294,0.000000,0.020622,0.000000,0.000000
f__2-12-FULL-35-15,0.000000,0.000000,0.052692,0.059694,0.015635,0.000000,0.020646,0.037927,0.000000,0.000000,2.152988,0.000000,0.000000,0.000000,0.000000,0.018336,0.000000,0.000000,0.000000,0.014775


# Merging host prediction with life mode prediction

In [13]:
life_mode_df = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/votus_definition/life_mode_pred/scaffolds_life_mode_pred.tsv',sep='\t')


host_plus_lifemode = pd.merge(iphop_df, life_mode_df, left_on='Virus', right_on='Accession', how='left')
host_plus_lifemode = host_plus_lifemode[host_plus_lifemode['Accession'].notna()]
host_plus_lifemode = host_plus_lifemode.drop(['Unnamed: 0','Accession'],axis=1)
host_plus_lifemode['sample'] = host_plus_lifemode['Virus'].apply(get_sample_name)
host_plus_lifemode['city'] = host_plus_lifemode['Virus'].apply(define_city)
#host_plus_lifemode[host_plus_lifemode['sample'] == 'BP4']['final_classification'].value_counts()
#host_plus_lifemode[host_plus_lifemode['city'] == 'Bragança']['final_classification'].value_counts()
host_plus_lifemode_grouped = host_plus_lifemode.groupby(['Family','final_classification']).count()['sample'].to_frame()
host_plus_lifemode_grouped = host_plus_lifemode_grouped.reset_index()

top15 = final_tax_by_sample_top15.head(14) #Sim só as 14. Sò essas aperecem no grafico
host_plus_lifemode_grouped = host_plus_lifemode_grouped[host_plus_lifemode_grouped['Family'].isin(top15.index.str.strip())]
host_plus_lifemode_grouped.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/04_life_mode/scaffolds_life_mode_pred.tsv',sep='\t')
host_plus_lifemode_grouped


,Family,final_classification,sample
10,f__Aestuariivirgaceae,Litic,1
17,f__Bacteroidaceae,Litic,11
19,f__Beijerinckiaceae,Litic,3
20,f__Beijerinckiaceae,Lysogenic,1
25,f__Burkholderiaceae,Litic,49
26,f__Burkholderiaceae,Lysogenic,18
37,f__Cyanobiaceae,Litic,36
38,f__Cyanobiaceae,Lysogenic,3
43,f__Enterobacteriaceae,Litic,10
44,f__Enterobacteriaceae,Lysogenic,11


In [20]:
host_plus_lifemode_sankey = host_plus_lifemode[(host_plus_lifemode['Family'] == 'f__Enterobacteriaceae')][['Family','sample','final_classification']]
host_plus_lifemode_sankey1 = host_plus_lifemode_sankey[['Family','sample']]
host_plus_lifemode_sankey1['step_from'] = 0
host_plus_lifemode_sankey1['step_to'] = 1
host_plus_lifemode_sankey1

,Family,sample,step_from,step_to
3,f__Enterobacteriaceae,BP4,0,1
45,f__Enterobacteriaceae,BP4,0,1
57,f__Enterobacteriaceae,BP4,0,1
58,f__Enterobacteriaceae,BP4,0,1
59,f__Enterobacteriaceae,BP4,0,1
70,f__Enterobacteriaceae,BP4,0,1
81,f__Enterobacteriaceae,BP4,0,1
99,f__Enterobacteriaceae,BP4,0,1
122,f__Enterobacteriaceae,BP4,0,1
169,f__Enterobacteriaceae,BP4,0,1


In [22]:
host_plus_lifemode_sankey2 = host_plus_lifemode_sankey[['sample','final_classification']]
host_plus_lifemode_sankey2['step_from'] = 1
host_plus_lifemode_sankey2['step_to'] = 2
host_plus_lifemode_sankey2 = host_plus_lifemode_sankey2.rename(columns={'sample':'Family','final_classification':'sample'})
host_plus_lifemode_sankey2

,Family,sample,step_from,step_to
3,BP4,Litic,1,2
45,BP4,Lysogenic,1,2
57,BP4,Lysogenic,1,2
58,BP4,Lysogenic,1,2
59,BP4,Lysogenic,1,2
70,BP4,Litic,1,2
81,BP4,Lysogenic,1,2
99,BP4,Lysogenic,1,2
122,BP4,Lysogenic,1,2
169,BP4,Litic,1,2


In [40]:
host_plus_lifemode_sankey_final = pd.concat([host_plus_lifemode_sankey1, host_plus_lifemode_sankey2])
host_plus_lifemode_sankey_final_final = host_plus_lifemode_sankey_final.groupby(['Family','sample','step_from', 'step_to']).value_counts().to_frame().reset_index()
host_plus_lifemode_sankey_final_final.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/votus_definition/dramv_results/host_plus_lifemode_sankey_final_final.tsv',sep='\t',index=False)
host_plus_lifemode_sankey_final_final

,Family,sample,step_from,step_to,count
0,AP3,Litic,1,2,1
1,BP4,Litic,1,2,7
2,BP4,Lysogenic,1,2,11
3,MP1,Litic,1,2,1
4,MP5,Litic,1,2,1
5,f__Enterobacteriaceae,AP3,0,1,1
6,f__Enterobacteriaceae,BP4,0,1,18
7,f__Enterobacteriaceae,MP1,0,1,1
8,f__Enterobacteriaceae,MP5,0,1,1


In [15]:
host_plus_lifemode[(host_plus_lifemode['city'] == 'Paragominas') & (host_plus_lifemode['sample'] == 'PP5')]['final_classification'].value_counts()

final_classification
Litic        10
Lysogenic     1
Name: count, dtype: int64